In [6]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))

from pathlib import Path
import torch
from torch.utils.data import DataLoader
from pneumonai.training.model import build_model
from pneumonai.training.dataset import ChestXrayDataset
from pneumonai.preprocessing.specification import load_preprocessing_spec

REPO = Path("..").resolve()
CHECKPOINT = REPO / "checkpoints" / "resnet18_lr0.001_best.pt"
SPEC = load_preprocessing_spec(REPO / "configs" / "preprocessing.yaml")
device = torch.device("cpu")

test_dataset = ChestXrayDataset(REPO / "data" / "splits" / "test.csv", SPEC)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)
val_dataset = ChestXrayDataset(REPO / "data" / "splits" / "validation.csv", SPEC)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

model = build_model("resnet18", num_classes=1).to(device)
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model.eval()
def remap_path(p: str) -> str:
    if p.startswith("/mnt/c/"):
        return p.replace("/mnt/c/", "C:/", 1)
    return p

test_dataset.df["image_path"] = test_dataset.df["image_path"].apply(remap_path)
val_dataset.df["image_path"] = val_dataset.df["image_path"].apply(remap_path)


In [7]:
probs = []
labels = []
with torch.no_grad():
    for inputs, targets in val_loader:
        logits = model(inputs).squeeze(1)
        probs.extend(torch.sigmoid(logits).tolist())
        labels.extend(targets.tolist())


In [8]:
from sklearn.metrics import roc_curve, roc_auc_score
import numpy as np

fpr, tpr, thresholds = roc_curve(labels, probs)
idx = np.argmax(tpr - fpr)
best_threshold = thresholds[idx]
auc = roc_auc_score(labels, probs)
print(f"Val AUC: {auc:.4f}")
print(f"Best threshold: {best_threshold:.4f}")



Val AUC: 0.8614
Best threshold: 0.3078


In [9]:
test_probs = []
test_labels = []
with torch.no_grad():
    for inputs, targets in test_loader:
        logits = model(inputs).squeeze(1)
        test_probs.extend(torch.sigmoid(logits).tolist())
        test_labels.extend(targets.tolist())

preds = (np.array(test_probs) >= best_threshold).astype(int)
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score
)
tn, fp, fn, tp = confusion_matrix(test_labels, preds).ravel()
specificity = tn / (tn + fp)


In [10]:
print(f"Test AUC:         {roc_auc_score(test_labels, test_probs):.4f}")
print(f"Sensitivity:      {tp / (tp + fn):.4f}")
print(f"Specificity:      {specificity:.4f}")
print(f"Precision:        {precision_score(test_labels, preds):.4f}")
print(f"Recall:           {recall_score(test_labels, preds):.4f}")
print(f"F1:               {f1_score(test_labels, preds):.4f}")
print(f"Threshold used:   {best_threshold:.4f}")
print(f"TP={tp} TN={tn} FP={fp} FN={fn}")


Test AUC:         0.8741
Sensitivity:      0.7439
Specificity:      0.8165
Precision:        0.5411
Recall:           0.7439
F1:               0.6265
Threshold used:   0.3078
TP=671 TN=2532 FP=569 FN=231


In [11]:
import yaml

report = {
    "model": "resnet18",
    "checkpoint": "checkpoints/resnet18_lr0.001_best.pt",
    "threshold": float(best_threshold),
    "test_auc": float(roc_auc_score(test_labels, test_probs)),
    "sensitivity": float(tp / (tp + fn)),
    "specificity": float(specificity),
    "precision": float(precision_score(test_labels, preds)),
    "recall": float(recall_score(test_labels, preds)),
    "f1": float(f1_score(test_labels, preds)),
    "confusion_matrix": {"tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)},
}

(REPO / "reports").mkdir(exist_ok=True)
with open(REPO / "reports" / "evaluation_report.yaml", "w") as f:
    yaml.dump(report, f, default_flow_style=False)

print("Saved.")


Saved.
